# Diffusion Transformer (DiT)
"Scalable Diffusion Models with Transformers" (Peebles & Xie, 2022)

**Idea:** keep the DDPM diffusion recipe, but replace the UNet noise predictor with a plain **ViT** over image patches, conditioned on the timestep and class label through **adaptive LayerNorm (adaLN-Zero)**.

### DiT vs Transformer vs ViT vs UNet-DDPM
| | Transformer (Vaswani 2017) | ViT (Dosovitskiy 2020) | DDPM (Ho 2020, UNet) | DiT |
|---|---|---|---|---|
| Task | seq-to-seq (translation) | classification | generation (denoising) | generation (denoising) |
| Input tokens | word ids -> `nn.Embedding` | image patches -> linear | whole noisy image (conv feature maps, no tokens) | **noisy** image patches -> linear |
| Backbone | encoder + decoder | encoder only | conv UNet (ResBlocks + some attention at low res) | encoder only (no `[CLS]`, no mask) |
| Conditioning | cross-attention on source sentence | none | timestep emb added inside each ResBlock | **adaLN-Zero**: timestep + class emb regress per-block shift / scale / gate |
| Norm | Post-LN | Pre-LN (learned $\gamma, \beta$) | GroupNorm | Pre-LN **without** learned affine; $\gamma, \beta$ come from the condition |
| Output | next-token distribution | one class from `[CLS]` | noise $\hat\epsilon$, same shape as image | per-token linear -> **unpatchify** -> noise $\hat\epsilon$ |
| Loss | cross-entropy | cross-entropy | MSE on noise | MSE on noise |

Shape flow: `x_t (N, 1, 32, 32) -> patches (N, 64, D) -> +pos -> DiT block x L (cond c: (N, D)) -> final adaLN -> linear (N, 64, 16) -> unpatchify -> eps (N, 1, 32, 32)`

**Simplification vs the paper:** the paper runs DiT in the **latent space of a pretrained VAE** (256x256x3 image -> 32x32x4 latent). Here there is **no VAE**: we diffuse directly on 32x32 MNIST pixels, which are already tiny.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Noise schedule + forward process (DDPM)
Forward process: add a little Gaussian noise at each of $T$ steps (formula below).

Because Gaussians compose, we can jump straight to any step in closed form (formula below). Training samples a random $t$, builds $x_t$ in one shot, and asks the model to predict $\epsilon$.

Linear $\beta$ schedule from $10^{-4}$ to $0.02$ over $T=1000$ steps (same as DDPM / DiT).

### Formula: Forward (noising) process
$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\left(x_t;\ \sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I\right)$$
- $x_0$: clean image, scaled to $[-1, 1]$, shape $1 \times 32 \times 32$
- $x_t$: image after $t$ noising steps; $x_T \approx \mathcal{N}(0, I)$
- $t \in \{1, \dots, T\}$: timestep, $T = 1000$ (code indexes $0..999$)
- $\beta_t$: noise variance added at step $t$, linear from $10^{-4}$ to $0.02$
- $\mathcal{N}(x;\ \mu, \Sigma)$: Gaussian density with mean $\mu$, covariance $\Sigma$; $I$: identity

### Formula: Closed-form $q(x_t \mid x_0)$
$$q(x_t \mid x_0) = \mathcal{N}\!\left(x_t;\ \sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t) I\right) \quad\Longleftrightarrow\quad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$$
$$\alpha_t = 1 - \beta_t, \qquad \bar\alpha_t = \prod_{s=1}^{t} \alpha_s$$
- $\alpha_t$: fraction of signal variance kept at step $t$
- $\bar\alpha_t$: total signal kept after $t$ steps (`alpha_bars`; $\approx 0.9999$ at $t=0$, $\approx 4 \cdot 10^{-5}$ at $t=999$)
- $\epsilon \sim \mathcal{N}(0, I)$: noise, same shape as $x_0$
- lets training jump to any $t$ in one step (`q_sample`)

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)  # (T,)
alphas = 1.0 - betas                                   # (T,)
alpha_bars = torch.cumprod(alphas, dim=0)              # (T,) signal kept after t steps

def q_sample(x0, t, noise):
    # (N, C, H, W), (N,), (N, C, H, W) -> x_t (N, C, H, W)
    ab = alpha_bars[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

print(f"alpha_bar: t=0 {alpha_bars[0]:.4f} | t=500 {alpha_bars[500]:.4f} | t=999 {alpha_bars[-1]:.6f}")

## 2. Patch embedding + attention (unchanged from ViT)
Same `PatchEmbedding` and `MultiHeadAttention` as the ViT notebook. The only change: `in_ch=1` (MNIST) and the input is a **noisy** image $x_t$.

### Formula: Scaled dot-product / multi-head attention
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V, \qquad \text{MSA}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\,W_O$$
- $X \in \mathbb{R}^{T \times D}$: tokens ($T = 64$, $D = 256$); $Q = XW_Q$, $K = XW_K$, $V = XW_V$
- $h$: heads (4); $d_k = D / h = 64$: per-head dim
- $\sqrt{d_k}$: keeps score variance ~1 so softmax does not saturate
- $W_O \in \mathbb{R}^{D \times D}$: output projection; no mask

In [ ]:
class PatchEmbedding(nn.Module):
    # (N, C, H, W) -> (N, num_patches, D)
    # e.g. (N, 1, 32, 32), patch 4 -> 8x8 = 64 patches -> (N, 64, D)
    def __init__(self, img_size, patch_size, in_ch, dim):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)                 # (N, D, 8, 8)
        x = x.flatten(2).transpose(1, 2) # (N, 64, D)
        return x


class MultiHeadAttention(nn.Module):
    # (N, T, D) -> (N, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim)   # Q, K, V in one matmul
        self.proj = nn.Linear(dim, dim)      # output projection W_O

    def forward(self, x):
        N, T, D = x.shape
        qkv = self.qkv(x).reshape(N, T, 3, self.num_heads, self.head_dim)  # (N, T, 3D) -> (N, T, 3, heads, head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)  # each (N, heads, T, head_dim)

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5  # (N, heads, T, T)
        att = att.softmax(dim=-1)            # no mask: every patch sees every patch

        out = att @ v                        # (N, heads, T, head_dim)
        out = out.transpose(1, 2).reshape(N, T, D)  # concat heads -> (N, T, D)
        return self.proj(out)                # (N, T, D)

## 3. Conditioning: timestep + class label embeddings
- **Timestep embedding:** the integer $t$ goes through the **same sin/cos formula as the Transformer's positional encoding** (here the "position" is the noise level), then a small MLP.
- **Label embedding:** a plain `nn.Embedding` with one **extra "null" class** (index 10). During training we replace the label with the null class 10% of the time (**label dropout**), so the same model also learns the *unconditional* noise prediction - needed for classifier-free guidance.
- The condition is just the sum of the two embeddings, shape `(N, D)` (formula below).

### Formula: Timestep embedding (sinusoidal)
$$\text{PE}(t)_{i} = \cos\!\left(t \cdot \omega_i\right),\quad \text{PE}(t)_{i + d/2} = \sin\!\left(t \cdot \omega_i\right), \qquad \omega_i = 10000^{-i/(d/2)}$$
$$\text{emb}(t) = W_2\,\text{SiLU}(W_1\,\text{PE}(t) + b_1) + b_2$$
- $t$: timestep (the "position" of the Transformer PE)
- $d$: frequency dim (256); $i = 0, \dots, 127$
- $\omega_i$: geometric range of frequencies from $1$ down to $\approx 10^{-4}$
- $W_1 \in \mathbb{R}^{D \times 256}$, $W_2 \in \mathbb{R}^{D \times D}$: MLP weights; $D = 256$
- $\text{SiLU}(x) = x \cdot \sigma(x)$, $\sigma$ = logistic sigmoid

### Formula: Conditioning vector
$$c = \text{emb}(t) + E_y[y]$$
- $E_y \in \mathbb{R}^{11 \times D}$: label embedding table (10 digits + 1 null row)
- $c \in \mathbb{R}^{D}$: one vector per image, shared by all 64 tokens and all blocks

In [ ]:
def sinusoidal_embedding(t, dim, max_period=10000):
    # (N,) -> (N, dim); same as Transformer PE with position = timestep
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device) / half)  # (half,)
    args = t.float()[:, None] * freqs[None]                                                # (N, half)
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)                          # (N, dim)


class TimestepEmbedder(nn.Module):
    # (N,) -> (N, D)
    def __init__(self, dim, freq_dim=256):
        super().__init__()
        self.freq_dim = freq_dim
        self.mlp = nn.Sequential(nn.Linear(freq_dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        return self.mlp(sinusoidal_embedding(t, self.freq_dim))  # (N, freq_dim) -> (N, D)


class LabelEmbedder(nn.Module):
    # (N,) -> (N, D); index num_classes is the "null" (unconditional) label
    def __init__(self, num_classes, dim, drop_prob=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.drop_prob = drop_prob
        self.embedding = nn.Embedding(num_classes + 1, dim)  # +1 for null class

    def forward(self, y):
        if self.training and self.drop_prob > 0:
            drop = torch.rand(y.shape[0], device=y.device) < self.drop_prob
            y = torch.where(drop, torch.full_like(y, self.num_classes), y)  # label dropout
        return self.embedding(y)  # (N, D)

## 4. DiT block with adaLN-Zero
ViT block: `x = x + MHA(LN(x))`, `x = x + MLP(LN(x))`, where LN has its own learned $\gamma, \beta$.

DiT block: LN has **no** learned affine. Instead an MLP on the condition $c$ regresses 6 vectors per block:
- **shift** $\beta$ and **scale** $\gamma$ -> `modulate(LN(x)) = LN(x) * (1 + scale) + shift` (adaptive LayerNorm, like FiLM)
- **gate** $\alpha$ -> scales the residual branch before it is added: `x = x + gate * MHA(...)`

One (shift, scale, gate) triple for the attention branch and one for the MLP branch.

**Zero init:** the `Linear` producing the 6 vectors starts at all zeros, so at init `gate = 0` and every block is the **identity function**. The network starts as "do nothing" and gradually opens each residual branch - the paper found this beats cross-attention and in-context conditioning, at negligible compute cost.

### Formula: adaLN (modulate)
$$(\beta_1, \gamma_1, \alpha_1, \beta_2, \gamma_2, \alpha_2) = W_{ada}\,\text{SiLU}(c) + b_{ada}$$
$$\text{adaLN}(h) = \text{LN}(h) \odot (1 + \gamma) + \beta$$
- $W_{ada} \in \mathbb{R}^{6D \times D}$, $b_{ada}$: per-block regression layer, **initialized to zero**
- $\beta$: shift, $\gamma$: scale (the "$1+$" makes $\gamma = 0$ mean "unchanged"); $\alpha$: gate
- index 1: attention branch; index 2: MLP branch
- $h \in \mathbb{R}^{T \times D}$: tokens ($T = 64$); $\beta, \gamma$ are broadcast over all tokens
- LN here has no learned $\gamma, \beta$ of its own (`elementwise_affine=False`)
- final layer uses the same thing with only $(\beta, \gamma)$, $W \in \mathbb{R}^{2D \times D}$

### Formula: Gated residual (adaLN-Zero DiT block)
$$h' = h + \alpha_1 \odot \text{MSA}\big(\text{adaLN}_1(h)\big), \qquad h_{out} = h' + \alpha_2 \odot \text{MLP}\big(\text{adaLN}_2(h')\big)$$
- $h$: block input $(T \times D)$; $h'$: after attention; $h_{out}$: block output
- $\alpha_1, \alpha_2 \in \mathbb{R}^D$: per-channel gates, broadcast over tokens
- zero init: $\alpha = 0$ -> $h_{out} = h$, every block starts as the identity
- MLP: $\text{Linear}(D \to 4D) \to \text{GELU} \to \text{Linear}(4D \to D)$; 6 blocks, 4 heads

### Formula: LayerNorm
$$\text{LN}(x) = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}, \qquad \mu = \frac{1}{D}\sum_{i=1}^{D} x_i,\quad \sigma^2 = \frac{1}{D}\sum_{i=1}^{D}(x_i - \mu)^2$$
- $x \in \mathbb{R}^D$: one token's features ($D = 256$), normalized per token
- $\mu, \sigma^2$: mean and variance over the features
- $\epsilon = 10^{-6}$: numerical stability
- no learned $\gamma, \beta$ here: adaLN supplies them from $c$

### Formula: GELU
$$\text{GELU}(x) = x\,\Phi(x) \approx 0.5\,x\left(1 + \tanh\!\left[\sqrt{2/\pi}\,\left(x + 0.044715\,x^3\right)\right]\right)$$
- $x$: scalar input (element-wise)
- $\Phi$: standard normal CDF
- the code uses the tanh approximation (`approximate="tanh"`), as in DiT

In [ ]:
def modulate(x, shift, scale):
    # (N, T, D), (N, D), (N, D) -> (N, T, D); same shift/scale for every token
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class DiTBlock(nn.Module):
    # x (N, T, D), c (N, D) -> (N, T, D)
    def __init__(self, dim, num_heads, mlp_ratio=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)  # gamma/beta come from c
        self.attn = MultiHeadAttention(dim, num_heads)
        self.ln2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_ratio * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(mlp_ratio * dim, dim),
        )
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))  # c -> 6 vectors
        nn.init.zeros_(self.adaLN_modulation[1].weight)  # adaLN-Zero: gate = 0 -> block = identity
        nn.init.zeros_(self.adaLN_modulation[1].bias)

    def forward(self, x, c):
        shift1, scale1, gate1, shift2, scale2, gate2 = self.adaLN_modulation(c).chunk(6, dim=1)  # each (N, D)
        x = x + gate1.unsqueeze(1) * self.attn(modulate(self.ln1(x), shift1, scale1))  # gate (N, 1, D) * (N, T, D)
        x = x + gate2.unsqueeze(1) * self.mlp(modulate(self.ln2(x), shift2, scale2))   # (N, T, D)
        return x

## 5. DiT
- `PatchEmbedding` + learned positional embedding, exactly like ViT (the paper uses fixed 2D sin/cos; learned is simpler and works as well here). No `[CLS]` token: we need an output for **every** patch.
- Final layer: adaLN (shift, scale only) -> `Linear(D, P*P*C)`, also zero-initialized, so the model initially predicts $\hat\epsilon = 0$.
- **Unpatchify:** inverse of patchify, each token's `P*P*C` values go back to its `P x P` spot in the image.
- The paper's model also predicts the variance $\Sigma$ (2x output channels); we keep the DDPM fixed variance and predict only $\epsilon$.

### Formula: Patchify / unpatchify
$$z_0 = [\,x_p^1 E;\ \dots;\ x_p^{N_p} E\,] + E_{pos}, \qquad N_p = \frac{HW}{P^2}, \qquad \hat\epsilon = \text{unpatchify}(\text{Linear}(\text{adaLN}(h_L)))$$
- $x_p^i \in \mathbb{R}^{P^2 C}$: $i$-th flattened patch of $x_t$; $P = 4$, $C = 1$ -> 16 values
- $N_p = 32 \cdot 32 / 16 = 64$ tokens; $E \in \mathbb{R}^{16 \times D}$ (the `Conv2d`); $E_{pos} \in \mathbb{R}^{64 \times D}$ learned
- $h_L$: output of the last block; Linear maps $D \to P^2 C = 16$
- unpatchify reshapes $(64, 16) \to (8, 8, 4, 4, 1) \to (1, 32, 32)$

In [ ]:
class FinalLayer(nn.Module):
    # x (N, T, D), c (N, D) -> (N, T, P*P*C)
    def __init__(self, dim, patch_size, out_ch):
        super().__init__()
        self.ln = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        self.linear = nn.Linear(dim, patch_size * patch_size * out_ch)
        for layer in (self.adaLN_modulation[1], self.linear):  # zero init -> predicts 0 at start
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)  # each (N, D)
        return self.linear(modulate(self.ln(x), shift, scale))  # (N, T, D) -> (N, T, P*P*C)


class DiT(nn.Module):
    # x_t (N, 1, 32, 32), t (N,), y (N,) -> eps_hat (N, 1, 32, 32)
    def __init__(self, img_size=32, patch_size=4, in_ch=1, num_classes=10,
                 dim=256, depth=6, num_heads=4, mlp_ratio=4, class_drop_prob=0.1):
        super().__init__()
        self.patch_size, self.in_ch = patch_size, in_ch
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_ch, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches, dim))
        self.t_embed = TimestepEmbedder(dim)
        self.y_embed = LabelEmbedder(num_classes, dim, class_drop_prob)
        self.blocks = nn.ModuleList([DiTBlock(dim, num_heads, mlp_ratio) for _ in range(depth)])
        self.final = FinalLayer(dim, patch_size, in_ch)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def unpatchify(self, x):
        # (N, T, P*P*C) -> (N, C, H, W)
        N, T, _ = x.shape
        P, C = self.patch_size, self.in_ch
        h = w = int(T ** 0.5)                         # 8 x 8 grid of patches
        x = x.reshape(N, h, w, P, P, C)               # (N, 8, 8, 4, 4, 1)
        x = x.permute(0, 5, 1, 3, 2, 4)               # (N, C, h, P, w, P)
        return x.reshape(N, C, h * P, w * P)          # (N, C, 32, 32)

    def forward(self, x, t, y):
        x = self.patch_embed(x) + self.pos_embed      # (N, 64, D)
        c = self.t_embed(t) + self.y_embed(y)         # (N, D) condition
        for block in self.blocks:
            x = block(x, c)                           # (N, 64, D)
        x = self.final(x, c)                          # (N, 64, 16)
        return self.unpatchify(x)                     # (N, 1, 32, 32)

model = DiT().to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
x = torch.randn(2, 1, 32, 32, device=device)
print(model(x, torch.tensor([0, 999], device=device), torch.tensor([3, 7], device=device)).shape)

## 6. Data - MNIST
28x28 grayscale digits, 10 classes, 60k train images. Pad to 32x32 (so patch 4 gives an 8x8 grid) and scale pixels to $[-1, 1]$, the same range as the $\mathcal{N}(0, I)$ noise we end at.

No augmentation: flips would change digit identity, and we want to model the data distribution as is.

In [ ]:
tf = transforms.Compose([
    transforms.Pad(2),                     # 28x28 -> 32x32
    transforms.ToTensor(),                 # [0, 1]
    transforms.Normalize((0.5,), (0.5,)),  # [-1, 1]
])
train_ds = datasets.MNIST("../data", train=True, download=True, transform=tf)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

# forward process on one digit
x0 = train_ds[0][0].unsqueeze(0).to(device)       # (1, 1, 32, 32)
ts = [0, 50, 100, 200, 400, 600, 800, 999]
fig, axes = plt.subplots(1, len(ts), figsize=(2 * len(ts), 2))
for ax, t in zip(axes, ts):
    xt = q_sample(x0, torch.tensor([t], device=device), torch.randn_like(x0))
    ax.imshow(xt[0, 0].cpu().clamp(-1, 1), cmap="gray")
    ax.set_title(f"t={t}")
    ax.axis("off")
plt.show()

## 7. Training
Each step: pick images $x_0$ with labels $y$, random $t \sim U\{0..T-1\}$, noise $\epsilon$, build $x_t$, and regress $\hat\epsilon = \text{DiT}(x_t, t, y)$ onto $\epsilon$ with **MSE** (DDPM's "simple" loss).

Recipe as in the paper: **AdamW**, constant LR, no weight decay, plus an **EMA** copy of the weights that is used for sampling (EMA weights give noticeably cleaner samples than the raw ones).

### Formula: Training loss (simple DDPM loss)
$$\mathcal{L} = \mathbb{E}_{x_0, y, t, \epsilon}\left[\ \left\lVert \epsilon - \epsilon_\theta(x_t, t, y) \right\rVert^2\ \right]$$
- $\epsilon_\theta$: the DiT, parameters $\theta$ (~7M); predicts the noise that was added
- $y$: class label (0-9), replaced by the null label $\varnothing$ (index 10) with probability $p_{\text{drop}} = 0.1$
- $t \sim U\{0, \dots, 999\}$: uniform random timestep per image
- $\lVert\cdot\rVert^2$: squared L2 norm; `F.mse_loss` averages it over pixels and batch (batch size 128)

### Formula: AdamW
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$
$$\theta_t = \theta_{t-1} - \eta\left(\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda\,\theta_{t-1}\right), \qquad \hat m_t = \frac{m_t}{1-\beta_1^t},\ \hat v_t = \frac{v_t}{1-\beta_2^t}$$
- $g_t = \nabla_\theta \mathcal{L}$: gradient at step $t$; $\theta_t$: parameters
- $m_t, v_t$: EMAs of gradient and squared gradient; $\hat m_t, \hat v_t$: bias-corrected
- $\beta_1, \beta_2 = 0.9, 0.999$ (defaults); $\epsilon = 10^{-8}$
- $\eta = 3 \cdot 10^{-4}$: constant learning rate
- $\lambda = 0$: no weight decay (as in DiT), so this is plain Adam here

### Formula: EMA of weights
$$\theta_{\text{EMA}} \leftarrow \mu\,\theta_{\text{EMA}} + (1 - \mu)\,\theta$$
- $\theta$: current model weights after each optimizer step
- $\mu = 0.999$: EMA decay (paper: 0.9999 over a much longer run); averages roughly the last $1/(1-\mu) = 1000$ steps
- $\theta_{\text{EMA}}$ (`ema`) is used only for sampling

In [ ]:
epochs = 20
lr = 3e-4
ema_decay = 0.999

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
ema = DiT().to(device)
ema.load_state_dict(model.state_dict())
ema.requires_grad_(False)

@torch.no_grad()
def update_ema(ema, model, decay):
    for pe, pm in zip(ema.parameters(), model.parameters()):
        pe.lerp_(pm, 1 - decay)  # pe = decay * pe + (1 - decay) * pm

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x0, y in train_loader:
        x0, y = x0.to(device), y.to(device)
        t = torch.randint(0, T, (x0.shape[0],), device=device)  # (N,)
        noise = torch.randn_like(x0)                            # (N, 1, 32, 32)
        xt = q_sample(x0, t, noise)                             # (N, 1, 32, 32)
        loss = F.mse_loss(model(xt, t, y), noise)  # eps_hat (N, 1, 32, 32) vs noise -> scalar

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        update_ema(ema, model, ema_decay)
        total_loss += loss.item()

    print(f"epoch {epoch+1:2d} | loss {total_loss / len(train_loader):.4f}")

## 8. Sampling with classifier-free guidance
Start from pure noise $x_T \sim \mathcal{N}(0, I)$ and run the DDPM reverse step $T$ times.

**Classifier-free guidance (CFG):** at each step run the model twice - with the real label $y$ and with the null label - and push the prediction away from the unconditional one (formula below). $w>1$ trades diversity for more class-typical digits. Both passes are done in one batch of size $2N$.

### Formula: Reverse (sampling) step
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat\epsilon\right) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I)$$
- $\hat\epsilon$: guided noise prediction at step $t$ (see CFG)
- first term: posterior mean, i.e. remove the predicted noise and rescale
- $\sigma_t^2 = \beta_t$: fixed DDPM variance (the paper learns $\Sigma_\theta$ instead)
- $z$: fresh noise; set to $0$ on the last step so $x_0$ is not re-noised
- repeated for $t = 999, \dots, 0$ starting from $x_T \sim \mathcal{N}(0, I)$

### Formula: Classifier-free guidance
$$\hat\epsilon = \epsilon_\theta(x_t, t, \varnothing) + w\,\big(\epsilon_\theta(x_t, t, y) - \epsilon_\theta(x_t, t, \varnothing)\big)$$
- $\epsilon_\theta(x_t, t, y)$: conditional prediction; $\epsilon_\theta(x_t, t, \varnothing)$: unconditional (null label)
- $w$: guidance scale (`cfg_scale = 3.0`); $w = 1$ -> plain conditional, $w = 0$ -> unconditional
- one model gives both thanks to label dropout during training
- intuition: $\epsilon$ is a scaled score $-\sqrt{1-\bar\alpha_t}\,\nabla \log p(x_t)$, so the difference is $\propto \nabla \log p(y \mid x_t)$, an implicit classifier gradient amplified by $w$

In [ ]:
@torch.no_grad()
def sample(model, y, cfg_scale=3.0):
    # y (N,) -> x_0 (N, 1, 32, 32)
    model.eval()
    N = y.shape[0]
    x = torch.randn(N, 1, 32, 32, device=device)          # x_T
    y_null = torch.full_like(y, model.y_embed.num_classes)
    for t in reversed(range(T)):
        tt = torch.full((2 * N,), t, device=device)          # (2N,)
        eps_cond, eps_uncond = model(torch.cat([x, x]), tt, torch.cat([y, y_null])).chunk(2)  # (2N, 1, 32, 32) -> 2 x (N, 1, 32, 32)
        eps = eps_uncond + cfg_scale * (eps_cond - eps_uncond)  # CFG

        mean = (x - betas[t] / (1 - alpha_bars[t]).sqrt() * eps) / alphas[t].sqrt()  # (N, 1, 32, 32)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)  # no noise on last step
        x = mean + betas[t].sqrt() * noise
    return x.clamp(-1, 1)

n_per_class = 8
y = torch.arange(10, device=device).repeat_interleave(n_per_class)  # (80,) 0,0,..,1,1,..
imgs = sample(ema, y, cfg_scale=3.0)                                 # (80, 1, 32, 32)

fig, axes = plt.subplots(10, n_per_class, figsize=(n_per_class, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i, 0].cpu(), cmap="gray")
    ax.axis("off")
for c in range(10):
    axes[c, 0].set_title(str(c), loc="left", fontsize=8)
plt.tight_layout()
plt.show()

## Notes
- **No VAE here.** DiT in the paper is a *latent* diffusion model: images are encoded by a frozen Stable-Diffusion VAE (8x downsampling) and DiT denoises the 32x32x4 latents. MNIST at 32x32x1 is already small, so we diffuse pixels directly; the backbone code is identical either way.
- **What changed vs ViT:** input is a noisy image, no `[CLS]`, output is one noise patch per token (unpatchify), and LayerNorm's $\gamma, \beta$ are predicted from $(t, y)$ instead of learned constants (adaLN-Zero). Attention and the patch embedding are untouched.
- **Scaling:** the paper's main result is that FID improves steadily with **Gflops** (more depth/width, or smaller patches = more tokens), not with parameter count alone. DiT-XL/2 (675M params, patch 2) beat all prior diffusion models on ImageNet 256x256.
- This model: ~7M params, 64 tokens; after ~20 epochs digits are clearly recognizable and match their row's class. Loss plateaus around ~0.03 quickly; sample quality keeps improving after the loss looks flat.
- **Sampling cost:** 1000 steps x 2 forward passes (CFG). Faster samplers (DDIM, 50-250 steps) plug in without retraining; the paper uses 250 DDPM steps with a learned variance.
- CFG scale: try `cfg_scale=1.0` (no guidance, more varied/sloppier digits) vs `5.0` (thicker, very "typical" digits).